# Qwen3-0.6B-Base × OpenWebMath — polar-family LoRA sweep (tanya-style)

Faithful replication of `tanya_results/owm300m_polar_sweep.md`: **continued pretraining** of
`Qwen/Qwen3-0.6B-Base` on the OpenWebMath corpus (all-token next-token loss, no prompt masking),
LoRA r=64, constant LR, single seed, 9000 steps. Data: `data/openwebmath_qwen3_320m_packed_seq2048`
(contiguous 2048-windows; `scripts/data/prepare_openwebmath_pretrain.py`). α = r = 64 (repo
convention; deviates from tanya's α=1.0).

Optimizer arms (tanya → repo-canonical mapping; iMuon dropped):

| tanya arm | this sweep |
|---|---|
| Frank-Wolfe | chord-tight-clean **ns=8 k=1** |
| BCD-Polar (k=3) | chord-tight-clean **ns=8 k=2** |
| AdamW | AdamW |
| iMuon | *(dropped)* |
| — | KL-diag +polar *(added)* |

Runs are discovered from the shared registry (`lora_playground.workloads`) — the same source the
leaderboard doc uses — so this cell never drifts from the data. `leaderboard_panel` renders
partial (in-flight) runs from their first eval; diverged LRs draw as hollow markers.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lora_playground').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.plotting import leaderboard_panel

# The four arms, by canonical label. chord-tight-clean k1/k2 share the ns=8 prefix.
def keep(label, cfg):
    return (
        label == 'AdamW'
        or label.startswith('chord-tight-clean ns=8 k=')
        or label == 'KL-diag +polar (f=10, δ=1e-4)'
    )

In [ ]:
_fig, _tdf, sdf = leaderboard_panel(
    'Qwen/Qwen3-0.6B-Base', 'openwebmath', 64,
    'Qwen3-0.6B × OpenWebMath × r=64 × 9000 steps — polar-family (tanya-style)',
    label_filter=keep,
    figsize=(12, 4.5),
)
plt.show()
sdf